In [9]:
import polars as pl
import numpy as np
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.svm import LinearSVC


Grid search dos melhores parametros

In [13]:

file = "/content/drive/MyDrive/NLP/train_arcaico_moderno.csv"
# file = "/content/drive/MyDrive/NLP/train_complexo_simples.csv"
# file = "/content/drive/MyDrive/NLP/train_literal_dinamico.csv"

csv = pl.read_csv(source=file, separator=";").filter(pl.col("text").is_not_null())

label_text = list(set(csv["style"]))
labels = [0 if style == label_text[0] else 1 for style in csv["style"]]
texts = [text for text in csv['text']]

pipeline = Pipeline([
    ("features", FeatureUnion([
        ("word", TfidfVectorizer(analyzer="word", ngram_range=(1,2), min_df=3)),
        ("char", TfidfVectorizer(analyzer="char", ngram_range=(1,5), min_df=3))
    ])),
    ("kbest", SelectKBest(chi2, k=22000)),
    ("clf", LinearSVC())
])


param_grid = {
    "kbest__k": [10000, 15000, 22000],
    "clf__C": [0.5, 0.9, 1.3],
}

grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=StratifiedKFold(n_splits=10, shuffle=True, random_state=42),
    scoring="accuracy",
    n_jobs=-1,
    verbose=2,
)

grid.fit(texts, labels)

print("\nBest parameters:", grid.best_params_)
print("Best cross-val accuracy:", grid.best_score_)
print()


best_model = grid.best_estimator_

Fitting 10 folds for each of 9 candidates, totalling 90 fits

Best parameters: {'clf__C': 0.9, 'kbest__k': 22000}
Best cross-val accuracy: 0.8903589421913892



Os melhores parametros sao k=22000 c=0.9

In [10]:
pipeline = Pipeline([
    ("features", FeatureUnion([
        ("word", TfidfVectorizer(analyzer="word", ngram_range=(1,2), min_df=3)),
        ("char", TfidfVectorizer(analyzer="char", ngram_range=(1,5), min_df=3))
    ])),
    ("kbest", SelectKBest(chi2, k=22000)),
    ("clf", LinearSVC(C=0.9))
])


In [11]:
files = ["/content/drive/MyDrive/NLP/train_arcaico_moderno.csv", "/content/drive/MyDrive/NLP/train_complexo_simples.csv", "/content/drive/MyDrive/NLP/train_literal_dinamico.csv"]

In [12]:
for file in files:
  csv = pl.read_csv(source=file, separator=";").filter(pl.col("text").is_not_null())

  label_text = list(set(csv["style"]))
  labels = [0 if style == label_text[0] else 1 for style in csv["style"]]
  texts = [text for text in csv['text']]

  scores = cross_val_score(pipeline, texts, labels, cv=10, scoring="accuracy")

  print(f"\nArquivo: {file}")
  print(f"Acurácias por fold: {np.round(scores, 4)}")
  print(f"Acurácia média (10 folds): {np.mean(scores):.4f}")


Arquivo: /content/drive/MyDrive/NLP/train_arcaico_moderno.csv
Acurácias por fold: [0.9    0.8967 0.9013 0.9049 0.8972 0.9035 0.8978 0.8983 0.898  0.8989]
Acurácia média (10 folds): 0.8997

Arquivo: /content/drive/MyDrive/NLP/train_complexo_simples.csv
Acurácias por fold: [0.9109 0.9063 0.9135 0.901  0.9087 0.9093 0.9072 0.9072 0.9013 0.9045]
Acurácia média (10 folds): 0.9070

Arquivo: /content/drive/MyDrive/NLP/train_literal_dinamico.csv
Acurácias por fold: [0.9024 0.8967 0.8956 0.8969 0.8966 0.8942 0.9048 0.8975 0.8972 0.9023]
Acurácia média (10 folds): 0.8984
